# Detrending Examples for Time Series Analysis
This notebook provides practical examples for different detrending techniques.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.tsa.stattools import kpss
import warnings

## A. Linear Trend

In [ ]:
# 1. Generate Data with a Linear Trend
t = np.arange(100)
trend = 0.5 * t
noise = np.random.normal(0, 2, 100)
y = trend + noise

# 2. Fit Linear Regression
model = LinearRegression()
model.fit(t.reshape(-1, 1), y)
trend_fitted = model.predict(t.reshape(-1, 1))

# 3. Detrend
y_detrended = y - trend_fitted
print("Linear detrending complete. First 5 detrended values:")
print(y_detrended[:5])

# 4. Plot Before and After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
ax1.plot(y, label='Original Data')
ax1.plot(trend_fitted, color='red', label='Fitted Trend')
ax1.set_title('Before Detrending (Linear Trend)')
ax1.legend()

ax2.plot(y_detrended, color='green', label='Detrended Data')
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('After Detrending')
ax2.legend()
plt.show()

## B. Polynomial Trend

In [ ]:
# 1. Generate Data with a Polynomial (Quadratic) Trend
t = np.arange(100)
trend = 0.05 * t**2 - 2 * t
noise = np.random.normal(0, 10, 100)
y = trend + noise

# 2. Fit Polynomial Regression (Degree 2)
coeffs = np.polyfit(t, y, 2)
trend_fitted = np.polyval(coeffs, t)

# 3. Detrend
y_detrended = y - trend_fitted
print("Polynomial detrending complete. First 5 detrended values:")
print(y_detrended[:5])

# 4. Plot Before and After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
ax1.plot(y, label='Original Data')
ax1.plot(trend_fitted, color='red', label='Fitted Trend')
ax1.set_title('Before Detrending (Polynomial Trend)')
ax1.legend()

ax2.plot(y_detrended, color='green', label='Detrended Data')
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('After Detrending')
ax2.legend()
plt.show()

## C. Changing / Non-Parametric Trend (MA & HP Filter)

In [ ]:
# 1. Generate Data with a Changing Trend (Sine wave + noise)
t = np.arange(100)
trend = np.sin(t / 10) * 10
noise = np.random.normal(0, 2, 100)
y = trend + noise
df = pd.DataFrame({'value': y})

# 2. Detrending using Moving Average (rolling window of 10)
df['moving_avg'] = df['value'].rolling(window=10, center=True).mean()
df['detrended_ma'] = df['value'] - df['moving_avg']

# 3. Detrending using HP Filter
cycle, trend_hp = sm.tsa.filters.hpfilter(df['value'], lamb=1600)
df['detrended_hp'] = cycle # The 'cycle' is the detrended data
print("Moving Average and HP Filter detrending complete.")

# 4. Plot Before and After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
ax1.plot(df['value'], label='Original Data', alpha=0.5)
ax1.plot(df['moving_avg'], color='red', label='Moving Average')
ax1.plot(df['value'] - trend_hp, color='orange', label='HP Filter', linestyle='--')
ax1.set_title('Before Detrending (Changing Trend)')
ax1.legend()

ax2.plot(df['detrended_hp'], color='green', label='Detrended (HP Filter)', alpha=0.7)
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('After Detrending')
ax2.legend()
plt.show()

## D. Stochastic Trend (Random Walk)

In [ ]:
# 1. Generate a Random Walk (Stochastic Trend)
np.random.seed(42)
steps = np.random.normal(0, 1, 100)
y = np.cumsum(steps) # Each value depends on the previous
df = pd.DataFrame({'value': y})

# 2. Detrending using Differencing
df['detrended_diff'] = df['value'].diff() # Y_t - Y_{t-1}
df_clean = df.dropna()
print("Differencing complete. First 5 differenced values:")
print(df_clean['detrended_diff'].head().values)

# 3. Plot Before and After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
ax1.plot(df['value'], label='Original Data')
ax1.set_title('Before Detrending (Random Walk)')
ax1.legend()

ax2.plot(df['detrended_diff'], color='green', label='Detrended Data (Differenced)')
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('After Detrending')
ax2.legend()
plt.show()

## Step 2: Statistical Verification (The KPSS Test)

In [ ]:
# Generate some sample data (e.g., a random walk)
np.random.seed(42)
steps = np.random.normal(0, 1, 100)
y = np.cumsum(steps)

# Run the KPSS test
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    statistic, p_value, n_lags, critical_values = kpss(y, regression='c')

print(f"KPSS p-value: {p_value}")
if p_value < 0.05:
    print("The series is NOT stationary (A trend exists, or it's a random walk).")
else:
    print("The series is stationary (No trend exists).")